In [1]:
import sys
sys.path.append("/home/suvendu/mlbd/project1/CSL7100-Project/code")  # adjust path if needed

from etl import run_etl


In [2]:
from config import print_config
from etl import run_etl

print_config()
run_etl()

🔧 CONFIGURATION
PROJECT_ROOT: /home/suvendu/mlbd/project1/CSL7100-Project
USE_HDFS: False
BASE_PATH: file:///home/suvendu/mlbd/project1/CSL7100-Project
PATHS:
  raw: file:///home/suvendu/mlbd/project1/CSL7100-Project/raw
  parquet: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet
  graph: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet/graph
  checkpoint: file:///home/suvendu/mlbd/project1/CSL7100-Project/checkpoints
  spark_temp: file:///home/suvendu/mlbd/project1/CSL7100-Project/spark-temp
💻 Running in LOCAL mode


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 11:02:36 WARN Utils: Your hostname, suvendu2023, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/17 11:02:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 11:02:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


📥 Loading data from: file:///home/suvendu/mlbd/project1/CSL7100-Project/raw
✅ Removed nulls


💾 Saving to: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet


✅ ETL Completed


In [3]:
#Data Validation (Quick sanity check)

In [4]:
from pyspark.sql import SparkSession
from config import PATHS

spark = SparkSession.builder.getOrCreate()

#df = spark.read.parquet(PATHS["parquet"])
df = spark.read.parquet(PATHS["parquet"])


df.show(5)
df.printSchema()
print("Total records:", df.count())

+--------------+----------+-------+--------------+
|    reviewerID|      asin|overall|unixReviewTime|
+--------------+----------+-------+--------------+
|A1738QAQKEW8O5|076780192X|    5.0|    1467590400|
| ABKAPN87T5G80|6300183513|    5.0|    1105142400|
|A34CTR6XZV65EC|0792110188|    5.0|    1366156800|
|A2NCBELIXEB677|6301767918|    5.0|    1469750400|
|A1S8BBGQQLGC01|1562550888|    5.0|    1322784000|
+--------------+----------+-------+--------------+
only showing top 5 rows
root
 |-- reviewerID: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- overall: double (nullable = true)
 |-- unixReviewTime: long (nullable = true)

Total records: 170487


In [5]:
print("Total records:", df.count())

Total records: 170487


In [6]:
#Basic Statistics
df.describe().show()

26/04/17 11:03:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 8:===>                                                     (1 + 15) / 16]

+-------+--------------------+--------------------+-----------------+--------------------+
|summary|          reviewerID|                asin|          overall|      unixReviewTime|
+-------+--------------------+--------------------+-----------------+--------------------+
|  count|              170487|              170487|           170487|              170487|
|   mean|                NULL| 5.504778845089227E9|4.221882020329996|1.3849071502507522E9|
| stddev|                NULL|2.0427185188942034E9|1.165933048718914|1.1548281767446889E8|
|    min|A0009988MRFQ3TROTQPI|          0000143588|              1.0|           901929600|
|    max|       AZZZA9JS7UPHO|          B01HIUL6WU|              5.0|          1538352000|
+-------+--------------------+--------------------+-----------------+--------------------+



In [7]:
from filter_5core import run_5core

run_5core()

💻 Running in LOCAL mode
📥 Loading data from: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet
Initial records: 170487

🔁 Iteration 1
Records before filtering: 170487


26/04/17 11:04:53 WARN DAGScheduler: Broadcasting large task binary with size 1140.1 KiB
                                                                                


🔁 Iteration 2
Records before filtering: 77576



🔁 Iteration 3
Records before filtering: 71938



🔁 Iteration 4
Records before filtering: 71403


...


🔁 Iteration 6
Records before filtering: 71336



🔁 Iteration 7
Records before filtering: 71332



🔁 Iteration 8
Records before filtering: 71330



🔁 Iteration 9
Records before filtering: 71330
✅ Converged
💾 Saving filtered data to: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet/5core


✅ 5-core filtering completed


In [8]:
from encode_ids import run_encoding

run_encoding()

💻 Running in LOCAL mode


26/04/17 11:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

✅ ID Encoding Completed


In [9]:
from etl import create_spark_session

spark = create_spark_session()



df = spark.read.parquet(PATHS["parquet"] + "/encoded")

df.show(5)
df.printSchema()

💻 Running in LOCAL mode
+-------+-------+-------+--------------+
|user_id|item_id|overall|unixReviewTime|
+-------+-------+-------+--------------+
|   1269|   8911|    2.0|    1469232000|
|  16705|  12156|    5.0|    1459641600|
|   6031|   5872|    5.0|    1462838400|
|  15655|   5727|    5.0|    1443744000|
|   2642|  11481|    1.0|    1477267200|
+-------+-------+-------+--------------+
only showing top 5 rows
root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- overall: double (nullable = true)
 |-- unixReviewTime: long (nullable = true)



In [10]:
import graph_builder
import importlib

importlib.reload(graph_builder)

<module 'graph_builder' from '/home/suvendu/mlbd/project1/CSL7100-Project/code/graph_builder.py'>

In [11]:
from graph_builder import build_graph
from etl import user_based_split
df = spark.read.parquet(PATHS["parquet"] + "/encoded")

train_df, test_df = user_based_split(df)

vertices, edges = build_graph(spark, train_df)

vertices.show(5)
edges.show(5)

🔀 Performing user-based split (corrected)


Train: 67796, Test: 3534
Max user_id: 24886


Vertices: 37651
Edges: 135592
+---+----+
| id|type|
+---+----+
|148|user|
|463|user|
|471|user|
|496|user|
|833|user|
+---+----+
only showing top 5 rows
+---+-----+------+
|src|  dst|weight|
+---+-----+------+
|148|24966|   0.5|
|148|29948|   0.5|
|463|25236|   0.5|
|463|36531|   0.5|
|471|34623|   0.5|
+---+-----+------+
only showing top 5 rows


In [12]:
vertices_test, edges_test = build_graph(spark, test_df)
vertices_test.show(5)
edges_test.show(5)

Max user_id: 24859


Vertices: 5101
Edges: 7068
+----+----+
|  id|type|
+----+----+
|2122|user|
|4818|user|
|6336|user|
|6357|user|
|8389|user|
+----+----+
only showing top 5 rows
+----+-----+------+
| src|  dst|weight|
+----+-----+------+
|2122|36477|   1.0|
|4818|36412|   1.0|
|6336|25111|   1.0|
|6357|26438|   1.0|
|8389|34137|   1.0|
+----+-----+------+
only showing top 5 rows


In [13]:
from graph_builder import run_graph_builder
from pyspark.sql.functions import max as spark_max
max_user_id = train_df.agg(spark_max("user_id")).collect()[0][0]

run_graph_builder(spark, train_df, max_user_id, mode='train')
run_graph_builder(spark, test_df, max_user_id, mode='test')

Max user_id: 24886


26/04/17 11:15:05 WARN CacheManager: Asked to cache already cached data.
26/04/17 11:15:05 WARN CacheManager: Asked to cache already cached data.


Vertices: 37651
Edges: 135592
💾 Saving vertices...


💾 Saving edges...


✅ Graph Construction Completed for train
✅ Graph Construction Completed
Max user_id: 24859


26/04/17 11:15:10 WARN CacheManager: Asked to cache already cached data.
26/04/17 11:15:10 WARN CacheManager: Asked to cache already cached data.


Vertices: 5101
Edges: 7068
💾 Saving vertices...


💾 Saving edges...


[Stage 164:==============================================>     (179 + 16) / 200]

✅ Graph Construction Completed for test
✅ Graph Construction Completed


In [14]:
from etl import create_spark_session
from pyspark.sql.functions import col
spark = create_spark_session()

vertices = spark.read.parquet(PATHS["parquet"] + "/graph/vertices/train")
edges = spark.read.parquet(PATHS["parquet"] + "/graph/edges/train")

💻 Running in LOCAL mode


In [15]:
# 1. User exists
source_user = 148   # example from your edges output
vertices.filter(col("id") == source_user).show()

# 2. User is type=user
vertices.filter((col("id") == source_user) & (col("type") == "user")).show()

# 3. User has edges
edges.filter(col("src") == source_user).show()

# Check if weights sum to 1 per user (important for PPR)
from pyspark.sql.functions import sum

edges.groupBy("src").agg(sum("weight")).show(5)

+---+----+
| id|type|
+---+----+
|148|user|
+---+----+

+---+----+
| id|type|
+---+----+
|148|user|
+---+----+

+---+-----+------+
|src|  dst|weight|
+---+-----+------+
|148|24966|   0.5|
|148|29948|   0.5|
+---+-----+------+

+---+-----------+
|src|sum(weight)|
+---+-----------+
|148|        1.0|
|463|        1.0|
|471|        1.0|
|496|        1.0|
|833|        1.0|
+---+-----------+
only showing top 5 rows


In [16]:
source_user = edges.select("src").distinct().limit(1).collect()[0][0]

In [17]:
from ppr import personalized_pagerank 
#ranks = personalized_pagerank(spark, vertices, edges, source_user)
#ranks.orderBy("rank", ascending=False).show(10)

In [18]:
from ppr import personalized_pagerank,personalized_pagerank_optimized
from pyspark.sql import SparkSession
from config import PATHS

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setCheckpointDir("/tmp/spark-checkpoints")
# Load graph
vertices = spark.read.parquet(PATHS["parquet"] + "/graph/vertices/train")
edges = spark.read.parquet(PATHS["parquet"] + "/graph/edges/train")

# Choose a user
source_user = 10
ranks, top_items = personalized_pagerank_optimized(
    spark,
    vertices,
    edges,
    source_id=source_user,
    alpha=0.15,
    max_iter=20,
    top_n=10
)

# Show results
top_items.show()

'''ranks = personalized_pagerank(
    spark,
    vertices,
    edges,
    source_id=source_user,
    alpha=0.15,
    max_iter=10
)'''

ranks.orderBy("rank", ascending=False).show(10)

🚀 Starting Personalized PageRank
🔄 Making graph bidirectional
🎯 Source user: 10

🔁 Iteration 1


Max diff: 1.7000000000000002

🔁 Iteration 2


Max diff: 1.4449999999999996

🔁 Iteration 3


Max diff: 1.2282499999999996

🔁 Iteration 4


Max diff: 1.0440124999999998

🔁 Iteration 5
Max diff: 0.8874106249999998

🔁 Iteration 6


...

Max diff: 0.1747084382025032

🔁 Iteration 16


Max diff: 0.14850217247212794

🔁 Iteration 17
Max diff: 0.12622684660130873

🔁 Iteration 18


Max diff: 0.10729281961111248

🔁 Iteration 19


Max diff: 0.09119889666944564

🔁 Iteration 20


Max diff: 0.0775190621690288

🎯 Extracting Top-N recommendations
+-----+--------------------+----+
|   id|                rank|type|
+-----+--------------------+----+
|27221|  0.0613968756911876|item|
|27007|0.058526204235574364|item|
|35118|0.044636994561084345|item|
|33672| 0.04045839385718231|item|
|27178|0.009828685942698492|item|
|25907|0.007440384069011409|item|
|37468| 0.00696854705089373|item|
|34388|0.006414684120347138|item|
|33148|0.005248786177093553|item|
|29472|0.004316252092257936|item|
+-----+--------------------+----+

+-----+--------------------+
|   id|                rank|
+-----+--------------------+
|   10| 0.18178659847241574|
|27221|  0.0613968756911876|
|27007|0.058526204235574364|
|35118|0.044636994561084345|
|33672| 0.04045839385718231|
|18926| 0.02141341502482668|
|21850|0.018719712358299334|
| 2756|0.018063867022471997|
| 6138|0.015429713767232971|
|27178|0.009828685942698492|
+-----+--------------------+
only showing top 10 rows


In [19]:
# Get items already interacted by user
seen_items = edges.filter(edges.src == source_user) \
                  .select("dst") \
                  .distinct()

# Remove them from recommendations
top_items_filtered = top_items.join(
    seen_items,
    top_items.id == seen_items.dst,
    how="left_anti"
)

top_items_filtered.show()

+-----+--------------------+----+
|   id|                rank|type|
+-----+--------------------+----+
|27178|0.009828685942698492|item|
|25907|0.007440384069011409|item|
|37468| 0.00696854705089373|item|
|34388|0.006414684120347138|item|
|33148|0.005248786177093553|item|
|29472|0.004316252092257936|item|
+-----+--------------------+----+



In [20]:
from pyspark.sql.functions import collect_list, lit

ground_truth = test_df.groupBy("user_id").agg(
    collect_list("item_id").alias("actual_items")
)

users = test_df.select("user_id").distinct().collect()

In [21]:
len(users)

2318

In [22]:
# per user page rakning for test user - found too slow 

In [23]:
#vertices = vertices.cache()
#edges = edges.cache()
from ppr import personalized_pagerank,personalized_pagerank_optimized_test
vertices.count()
edges.count()

all_recommendations = []

for i, row in enumerate(users):
    user_id = row["user_id"]

    print(f"Running PPR for user: {user_id}")

    ranks, top_items = personalized_pagerank_optimized_test(
        spark,
        vertices,
        edges,
        source_id=user_id,
        alpha=0.15,
        max_iter=10,   # reduce iterations
        top_n=10
    )

    recs = top_items.select(
        col("id").alias("item_id"),
        col("rank")
    ).withColumn("user_id", lit(user_id))

    all_recommendations.append(recs)

    # 🔥 VERY IMPORTANT
    ranks.unpersist()
    top_items.unpersist()

    
    

Running PPR for user: 28
🚀 Starting Personalized PageRank


26/04/17 11:16:19 WARN CacheManager: Asked to cache already cached data.


🎯 Source user: 28

🔁 Iteration 1


Max diff: 0.85

🔁 Iteration 2


Max diff: 0.36124999999999996

🔁 Iteration 3


...

Max diff: 0.23821713789682536

🔁 Iteration 5


Max diff: 0.15942138711706472

🔁 Iteration 6


Max diff: 0.13550817904950502

🔁 Iteration 7


ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/home/suvendu/miniconda3/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/suvendu/miniconda3/lib/python3.11/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/suvendu/miniconda3/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# not used

In [24]:
from pyspark.sql.functions import col, explode, lit
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.functions import struct, collect_list, map_from_entries
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.types import MapType, DoubleType
from pyspark.sql.functions import udf

In [25]:
#VALID USER FILTER

In [26]:
graph_users = vertices.filter(col("type") == "user").select("id")

valid_users_df = graph_users.join(
    ground_truth.select(col("user_id").alias("id")),
    "id"
)

user_ids = [r["id"] for r in valid_users_df.collect()]
user_ids_set = set(user_ids)

print("Valid users:", len(user_ids))

Valid users: 2318


In [27]:
#🔹 Step 1: Initialize Users

In [28]:
# Use ONLY test users (critical for evaluation)
test_users = ground_truth.select("user_id").distinct()

user_ids = [r["user_id"] for r in test_users.collect()]
user_ids_set = set(user_ids)


In [29]:
#🔹 Step 2: Initialize Sparse Rank Map

In [30]:
def init_rank(node_id):
    if node_id in user_ids_set:
        return {float(node_id): 1.0}
    return {}

init_udf = udf(init_rank, MapType(DoubleType(), DoubleType()))

ranks = vertices.select("id").withColumn("rank_map", init_udf(col("id")))

In [31]:
init_ranks = ranks

In [32]:
#🔹 Step 3: Helper (Explode Map)

In [33]:
def explode_map(df):
    return df.select(
        col("id"),
        explode("rank_map").alias("user_id", "score")
    ).withColumn(
        "user_id", col("user_id").cast("int")
    )

In [34]:
edges = edges.select("src", "dst", "weight").cache()
edges.count()

26/04/17 11:18:55 WARN CacheManager: Asked to cache already cached data.


135592

In [35]:
#🔹 Step 5: Fast-PPR Iterations

In [36]:
alpha = 0.15
max_iter = 12  # 8

for i in range(max_iter):
    print(f"\nIteration {i+1}")

    e = explode_map(ranks).alias("e")
    g = edges.alias("g")

    # -------------------------
    # Propagation
    # -------------------------
    contribs = e.join(
        g,
        col("e.id") == col("g.src")
    ).select(
        col("g.dst").alias("id"),
        col("e.user_id"),
        (col("e.score") * col("g.weight")).alias("score")
    )

    # -------------------------
    # Aggregate
    # -------------------------
    agg = contribs.groupBy("id", "user_id") \
                  .agg(spark_sum("score").alias("score"))

    # -------------------------
    # Damping
    # -------------------------
    agg = agg.withColumn(
        "score",
        (1 - alpha) * col("score")
    )

    # -------------------------
    # ✅ Correct Teleport
    # -------------------------
    teleport = explode_map(init_ranks) \
        .withColumn("score", alpha * col("score"))

    new_ranks = agg.union(teleport)

    # -------------------------
    # Merge
    # -------------------------
    new_ranks = new_ranks.groupBy("id", "user_id") \
                         .agg(spark_sum("score").alias("score"))

    # -------------------------
    # 🔥 PRUNING (SAFE VALUE)
    # -------------------------
    window = Window.partitionBy("id").orderBy(col("score").desc())

    new_ranks = new_ranks.withColumn(
        "rnk", row_number().over(window)
    ).filter(col("rnk") <= 100).drop("rnk")   # ← increased from 20

    # -------------------------
    # Back to map
    # -------------------------
    ranks = new_ranks.groupBy("id").agg(
        map_from_entries(
            collect_list(struct("user_id", "score"))
        ).alias("rank_map")
    )

    # -------------------------
    # Checkpoint
    # -------------------------
    if i % 2 == 0:
        ranks = ranks.checkpoint()

    # -------------------------
    # DEBUG (IMPORTANT)
    # -------------------------
    print("Non-empty nodes:",
          ranks.selectExpr("size(rank_map) > 0").filter("`(size(rank_map) > 0)`").count())


Iteration 1


Non-empty nodes: 8678

Iteration 2


Non-empty nodes: 28526

Iteration 3


Non-empty nodes: 34443

Iteration 4


Non-empty nodes: 37074

Iteration 5


...

Non-empty nodes: 37601

Iteration 10


Non-empty nodes: 37601

Iteration 11


Non-empty nodes: 37601

Iteration 12


[Stage 1867:==================================================>   (16 + 1) / 17]

Non-empty nodes: 37601


In [37]:
#Step 6: Extract Recommendations (PER USER)

In [38]:
exploded = explode_map(ranks)

recommendations = exploded.select(
    col("user_id"),
    col("id").alias("item_id"),
    col("score").alias("rank")
)

In [39]:
#🔹 Step 7: Keep Only Items

In [40]:
recommendations = recommendations.join(
    vertices,
    recommendations.item_id == vertices.id
).filter(
    col("type") == "item"
).select(
    "user_id", "item_id", "rank"
)

In [41]:
#🔹 Step 8: Reverse Item Shift (IMPORTANT)

In [42]:
max_user_id = train_df.agg({"user_id": "max"}).collect()[0][0]

recommendations = recommendations.withColumn(
    "item_id",
    col("item_id") - (max_user_id + 1)
)

In [43]:
#🔹 Step 9: Remove Seen Items

In [44]:
from pyspark.sql.functions import collect_list, array_contains

train_items = train_df.groupBy("user_id").agg(
    collect_list("item_id").alias("train_items")
)

recommendations = recommendations.join(train_items, "user_id", "left")

recommendations = recommendations.filter(
    ~array_contains(col("train_items"), col("item_id"))
).drop("train_items")

In [45]:
#🔹 Step 10: Top-K per User

In [46]:
K = 5

window = Window.partitionBy("user_id").orderBy(col("rank").desc())

recommendations = recommendations.withColumn(
    "rnk", row_number().over(window)
).filter(col("rnk") <= K).drop("rnk")

In [47]:
#🔹 Step 11: Final Debug (MUST PASS)

In [48]:
print("Recommendations:", recommendations.count())

overlap = recommendations.select("user_id").distinct().intersect(
    ground_truth.select("user_id").distinct()
).count()

print("User overlap:", overlap)

26/04/17 11:21:35 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:21:36 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:21:40 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/17 11:21:43 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/17 11:21:45 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/17 11:21:47 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
                                                                                

Recommendations: 11549


26/04/17 11:21:49 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:21:49 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:21:51 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/17 11:21:54 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/17 11:21:56 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/17 11:21:58 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB


User overlap: 2312


In [49]:
#Step 8: FILTER ONLY TEST USERS (VERY IMPORTANT)

In [50]:
from evaluation import evaluate_precision_recall, compute_map

precision, recall = evaluate_precision_recall(
    recommendations,
    ground_truth,
    k=5
)

pred_items_df = recommendations.groupBy("user_id").agg(
    collect_list("item_id").alias("pred_items")
)

map_k = compute_map(pred_items_df, ground_truth, k=5)

print("\n🎯 FINAL RESULTS")
print(f"Precision@5: {precision}")
print(f"Recall@5: {recall}")
print(f"MAP@5: {map_k}")

26/04/17 11:22:00 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:22:00 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:22:01 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/17 11:22:04 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/17 11:22:06 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/17 11:22:08 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
                                                                                


✅ Precision@5: 0.002076124567474048
✅ Recall@5: 0.009252965892239248


26/04/17 11:22:10 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:22:10 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:22:11 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/17 11:22:14 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/17 11:22:15 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/17 11:22:17 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB


✅ MAP@5: 0.006069780853517877

🎯 FINAL RESULTS
Precision@5: 0.002076124567474048
Recall@5: 0.009252965892239248
MAP@5: 0.006069780853517877


In [51]:
print("Recommendations count:", recommendations.count())
recommendations.show(5)

26/04/17 11:22:18 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:22:18 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:22:21 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/17 11:22:24 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/17 11:22:26 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/17 11:22:28 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
                                                                                

Recommendations count: 11549


26/04/17 11:22:29 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:22:29 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/17 11:22:30 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/17 11:22:32 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/17 11:22:33 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
[Stage 2046:===>                                                  (1 + 16) / 17]

+-------+-------+--------------------+
|user_id|item_id|                rank|
+-------+-------+--------------------+
|     28|   1563|0.018325065396073644|
|     28|  10218|0.011242896528769536|
|     28|   6645|0.010670805953341786|
|     28|   5309|0.010123896899973448|
|     28|   5164|0.006546757273379857|
+-------+-------+--------------------+
only showing top 5 rows


26/04/17 11:22:35 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
                                                                                

In [52]:
print("Ground truth count:", ground_truth.count())
ground_truth.show(5)

Ground truth count: 2318
+-------+------------+
|user_id|actual_items|
+-------+------------+
|     28|      [8050]|
|     41|      [2477]|
|     45|      [3073]|
|     50|      [1858]|
|     64|     [11970]|
+-------+------------+
only showing top 5 rows


In [53]:
from pyspark.sql.functions import col, count
from pyspark.sql.functions import max as spark_max
from pyspark.sql.functions import min as spark_min
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import collect_list
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
import builtins

In [ ]:
# Hybrid model (PPR + Content) using: popularity + recency features 
# normalization
# DOT PRODUCT
# weighted user profile

In [54]:
# 🔹 Step 1: Build Item Features (Popularity + Recency)

In [55]:
# --- Popularity ---
item_popularity = train_df.groupBy("item_id").agg(
    count("*").alias("popularity")
)

max_pop = item_popularity.agg(spark_max("popularity")).collect()[0][0]

item_popularity = item_popularity.withColumn(
    "popularity",
    col("popularity") / max_pop
)

# --- Recency ---
max_time = train_df.agg(spark_max("unixReviewTime")).collect()[0][0]

item_recency = train_df.withColumn(
    "recency",
    col("unixReviewTime") / max_time
).select("item_id", "recency").dropDuplicates(["item_id"])

# --- Combine ---
item_features = item_popularity.join(item_recency, "item_id")

In [56]:
#Normalize

In [57]:
w = Window.partitionBy()

item_features = item_features \
    .withColumn("pop_norm",
        (col("popularity") - spark_min("popularity").over(w)) /
        (spark_max("popularity").over(w) - spark_min("popularity").over(w) + 1e-9)
    ) \
    .withColumn("rec_norm",
        (col("recency") - spark_min("recency").over(w)) /
        (spark_max("recency").over(w) - spark_min("recency").over(w) + 1e-9)
    )

In [58]:
#Step 3: Convert to Vector

In [59]:
assembler = VectorAssembler(
    inputCols=["pop_norm", "rec_norm"],
    outputCol="features"
)

item_features = assembler.transform(item_features).select("item_id", "features")

In [60]:
# Step 4: Build USER PROFILE (Weighted)

In [61]:
def weighted_avg(vectors, ratings):
    if not vectors:
        return None

    vectors = [list(v) for v in vectors]
    total = builtins.sum(ratings)

    dim = len(vectors[0])
    avg = [0.0] * dim

    for v, r in zip(vectors, ratings):
        for i in range(dim):
            avg[i] += v[i] * r

    return Vectors.dense([x / total for x in avg])

weighted_avg_udf = udf(weighted_avg, VectorUDT())

user_profiles = train_df.join(item_features, "item_id") \
    .groupBy("user_id") \
    .agg(
        collect_list("features").alias("feat_list"),
        collect_list("overall").alias("rating_list")
    ) \
    .withColumn("user_vec", weighted_avg_udf("feat_list", "rating_list")) \
    .select("user_id", "user_vec")

In [62]:
#Step 5: DOT PRODUCT (FIXED SIMILARITY)

In [63]:
def dot_product(v1, v2):
    if v1 is None or v2 is None:
        return 0.0

    v1 = list(v1)
    v2 = list(v2)

    return float(builtins.sum(a*b for a,b in zip(v1, v2)))

dot_udf = udf(dot_product, DoubleType())

In [64]:
# Step 6: Compute Content Score

In [65]:
content_scores = recommendations.join(user_profiles, "user_id") \
    .join(item_features, "item_id") \
    .withColumn(
        "content_score",
        dot_udf(col("user_vec"), col("features"))
    ).select("user_id", "item_id", "content_score")

In [66]:
#Step 7: Normalize Scores

In [67]:
w = Window.partitionBy("user_id")

# PPR normalization
ppr_norm = recommendations.withColumn(
    "ppr_norm",
    (col("rank") - spark_min("rank").over(w)) /
    (spark_max("rank").over(w) - spark_min("rank").over(w) + 1e-9)
)

# Content normalization
content_norm = content_scores.withColumn(
    "content_norm",
    (col("content_score") - spark_min("content_score").over(w)) /
    (spark_max("content_score").over(w) - spark_min("content_score").over(w) + 1e-9)
)

In [68]:
# 🔹 Step 8: Combine (Hybrid Score)

In [69]:
hybrid = ppr_norm.join(content_norm, ["user_id", "item_id"])

w1 = 0.7
w2 = 0.3

hybrid = hybrid.withColumn(
    "final_score",
    w1 * col("ppr_norm") + w2 * col("content_norm")
)

In [70]:
# 🔹 Step 9: Top-K Recommendations

In [71]:
from pyspark.sql.functions import row_number

K = 5

window = Window.partitionBy("user_id").orderBy(col("final_score").desc())

hybrid_recommendations = hybrid.withColumn(
    "rnk", row_number().over(window)
).filter(col("rnk") <= K).select(
    "user_id", "item_id", col("final_score").alias("rank")
)

In [72]:
# 🔹 Step 10: Evaluate

In [73]:
from evaluation import evaluate_precision_recall, compute_map
from pyspark.sql.functions import collect_list

precision, recall = evaluate_precision_recall(
    hybrid_recommendations,
    ground_truth,
    k=5
)

pred_items_df = hybrid_recommendations.groupBy("user_id").agg(
    collect_list("item_id").alias("pred_items")
)

map_k = compute_map(pred_items_df, ground_truth, k=5)

print("\n🎯 HYBRID RESULTS (FIXED)")
print("Precision@5:", precision)
print("Recall@5:", recall)
print("MAP@5:", map_k)

26/04/17 11:23:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:23:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:23:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:23:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:23:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:23:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1


✅ Precision@5: 0.002076124567474049
✅ Recall@5: 0.009252965892239248


26/04/17 11:23:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:23:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:23:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:23:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:23:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:23:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

✅ MAP@5: 0.005942906574394463

🎯 HYBRID RESULTS (FIXED)
Precision@5: 0.002076124567474049
Recall@5: 0.009252965892239248
MAP@5: 0.005942906574394463


In [74]:
item_features.show(5)

26/04/17 11:24:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

+-------+--------------------+
|item_id|            features|
+-------+--------------------+
|   1591|[0.02958579878680...|
|   2142|[0.05917159757361...|
|   9900|[0.01183431951472...|
|   6658|[0.01183431951472...|
|   3175|[0.04733727805889...|
+-------+--------------------+
only showing top 5 rows


In [75]:
item_features.select("features").show(5, False)

26/04/17 11:24:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

+-----------------------------------------+
|features                                 |
+-----------------------------------------+
|[0.029585798786807185,0.4046089375416146]|
|[0.05917159757361438,0.8065642438049102] |
|[0.011834319514722873,0.7688547466919533]|
|[0.011834319514722873,0.8314245789386374]|
|[0.011834319514722873,0.9286312826075926]|
+-----------------------------------------+
only showing top 5 rows


26/04/17 11:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [76]:
content_scores.select("content_score").describe().show()

26/04/17 11:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 11:24:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

+-------+--------------------+
|summary|       content_score|
+-------+--------------------+
|  count|               11549|
|   mean|  0.5615608832890554|
| stddev| 0.20572041191752258|
|    min|0.004586391354828443|
|    max|  1.5692505611759227|
+-------+--------------------+



In [77]:
import graph_builder
import importlib

importlib.reload(graph_builder)

<module 'graph_builder' from '/home/suvendu/mlbd/project1/CSL7100-Project/code/graph_builder.py'>

In [78]:
# SPARK NODE2VEC (Word2Vec-based)
#Node2Vec:- Learns global graph structure - → captures similarity between users/items
# Random Walks + Word2Vec (Spark ML)
# Nodes appearing in similar walks → similar embeddings

In [79]:
!pip install node2vec --trusted-host pypi.org --trusted-host files.pythonhosted.org

In [80]:
# For Node2Vec (Random Walk + Word2Vec)
# STEP 1: Prepare Adjacency List- Generate Random Walks-

In [81]:
from graph_builder import build_graph_node2vec , generate_walks

# Step 1: Build graph  #  Prepare Adjacency List
adj, max_user_id = build_graph_node2vec(train_df)

# Step 2: Generate walks
walks = generate_walks(adj, num_walks=20, walk_length=25) # 

walks_df = spark.createDataFrame([(w,) for w in walks], ["walk"])

# Step 3: Word2Vec- Train Word2Vec
from pyspark.ml.feature import Word2Vec

word2vec = Word2Vec(
    vectorSize=128,  # increased from 64 
    windowSize=10,   # increased from 5
    minCount=1,
    inputCol="walk",
    outputCol="embedding"
)

model = word2vec.fit(walks_df)

# Step 4: Get Node Embeddings
embeddings = model.getVectors() \
    .withColumn("id", col("word").cast("int")) \
    .select("id", col("vector").alias("embedding"))

Max user_id: 24886


Adjacency nodes: 37651


26/04/17 11:26:58 WARN TaskSetManager: Stage 2332 contains a task of very large size (9150 KiB). The maximum recommended task size is 1000 KiB.
26/04/17 11:27:16 WARN TaskSetManager: Stage 2334 contains a task of very large size (9150 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

In [82]:
# Split Users and Items (Same as your logic)

In [83]:
user_embeddings = embeddings.filter(col("id") <= max_user_id) \
    .withColumnRenamed("id", "user_id")

item_embeddings = embeddings.filter(col("id") > max_user_id) \
    .withColumn(
        "item_id", col("id") - (max_user_id + 1)
    ).select("item_id", "embedding")

In [84]:
# Dot Product Similarity

In [85]:
import builtins
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def dot_product(v1, v2):
    if v1 is None or v2 is None:
        return 0.0
    return float(builtins.sum(a*b for a,b in zip(v1, v2)))

dot_udf = udf(dot_product, DoubleType())

In [86]:
# Candidate Filtering

In [87]:
top_items = train_df.groupBy("item_id").count() \
    .orderBy(col("count").desc()) \
    .limit(8000) \
    .select("item_id")

candidate_items = item_embeddings.join(top_items, "item_id")

In [88]:
# Generate Recommendations

In [89]:
recs = user_embeddings.alias("u") \
    .crossJoin(candidate_items.alias("i")) \
    .withColumn(
        "score",
        dot_udf(col("u.embedding"), col("i.embedding"))
    ).select(
        col("u.user_id"),
        col("i.item_id"),
        col("score")
    )

In [90]:
# Top-K per User

In [91]:
K = 5

window = Window.partitionBy("user_id").orderBy(col("score").desc())

recommendations = recs.withColumn(
    "rnk", row_number().over(window)
).filter(col("rnk") <= K).drop("rnk")

In [92]:
# Evaluate

In [93]:
# Build Ground Truth from test_df

In [94]:
from pyspark.sql.functions import collect_list

ground_truth = test_df.groupBy("user_id").agg(
    collect_list("item_id").alias("actual_items")
)

In [95]:
# Ensure Recommendations Format

In [96]:
recommendations = recommendations.select(
    "user_id", "item_id", col("score").alias("rank")
)

In [97]:
# DEBUG

In [ ]:
print("Recommendations count:", recommendations.count())
print("Ground truth count:", ground_truth.count())

common_users = recommendations.select("user_id").distinct() \
    .intersect(ground_truth.select("user_id").distinct())

print("Users in both:", common_users.count())

In [ ]:
# Run Evaluation

In [98]:
recommendations = recommendations.repartition(200)
ground_truth = ground_truth.repartition(200)

recommendations.cache()
ground_truth.cache()

DataFrame[user_id: int, actual_items: array<int>]

In [100]:
#spark.conf.set("spark.sql.shuffle.partitions", "200")
#spark.conf.set("spark.executor.memory", "6g")   # increase if possible
#spark.conf.set("spark.driver.memory", "4g")

In [105]:
from evaluation import evaluate_precision_recall, compute_map ,evaluate_precision_recall_safe, compute_map_safe
from pyspark.sql.functions import collect_list

precision, recall = evaluate_precision_recall_safe(
    recommendations,
    ground_truth,
    k=5
)

pred_items_df = recommendations.groupBy("user_id").agg(
    collect_list("item_id").alias("pred_items")
)

#map_k = compute_map_safe(recommendations, ground_truth, k=5)

print("\n🎯 SPARK NODE2VEC RESULTS")
print("Precision@5:", precision)
print("Recall@5:", recall)
#print("MAP@5:", map_k)


🎯 SPARK NODE2VEC RESULTS
Precision@5: 0.20000000000000004
Recall@5: 0.8690476190476191


In [106]:
map_k = compute_map_safe(recommendations, ground_truth, k=5)
print("MAP@5:", map_k)

                                                                                0]

MAP@5: 0.5321428571428571


In [102]:
user_profiles.printSchema()
item_features.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- user_vec: vector (nullable = true)

root
 |-- item_id: integer (nullable = true)
 |-- features: vector (nullable = true)



In [103]:
recommendations.printSchema()
recommendations.show(5)

root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- rank: double (nullable = true)

+-------+-------+------------------+
|user_id|item_id|              rank|
+-------+-------+------------------+
|  17193|   1464| 22.43317426678341|
|   5692|   5123|13.579652126658457|
|   7567|     42|17.109225040516577|
|   1395|   7410|15.007991150302063|
|   2923|   2850|12.838124131119665|
+-------+-------+------------------+
only showing top 5 rows


In [104]:
recommendations.orderBy("user_id", col("rank").desc()).show(10)

+-------+-------+------------------+
|user_id|item_id|              rank|
+-------+-------+------------------+
|      1|   9642|  9.86451926640139|
|      1|  11103| 9.613378068109775|
|      1|  10830| 9.489117954991245|
|      1|  12177| 6.972467463969602|
|      1|   1818| 6.614793739786211|
|      2|   9315| 12.96996757941212|
|      2|  11108|11.873465324567828|
|      2|   9999|10.607034482978749|
|      2|   2361|10.531490071158549|
|      2|   9045| 8.101983980292916|
+-------+-------+------------------+
only showing top 10 rows


In [107]:
from pyspark.sql.functions import size

ground_truth.select(size("actual_items")).describe().show()

+-------+------------------+
|summary|size(actual_items)|
+-------+------------------+
|  count|              2318|
|   mean|1.5245901639344261|
| stddev| 1.516147919809068|
|    min|                 1|
|    max|                20|
+-------+------------------+



In [111]:
from pyspark.sql.functions import size

ground_truth_filtered = ground_truth.filter(size("actual_items") >= 2)

In [112]:
precision, recall = evaluate_precision_recall_safe(
    recommendations.join(ground_truth_filtered, "user_id"),
    ground_truth_filtered,
    k=5
)

map_k = compute_map_safe(
    recommendations.join(ground_truth_filtered, "user_id"),
    ground_truth_filtered,
    k=5
)

In [110]:
print("\n🎯 SPARK NODE2VEC RESULTS")
print("Precision@5:", precision)
print("Recall@5:", recall)
print("MAP@5:", map_k)


🎯 SPARK NODE2VEC RESULTS
Precision@5: None
Recall@5: None
MAP@5: None
